# Chapitre 3 - Statistiques d'implémentation

Ce notebook génère les tableaux et graphiques descriptifs utilisés pour documenter l'implémentation du système de recommandation hybride. Il lit les données normalisées, interroge pgvector et Neo4j, puis exporte les sorties vers `outputs/memoire_stats/chapter3` et `rapport/figures/generated/chapter3`.

Principe méthodologique : les chiffres du mémoire doivent provenir des artefacts réels du projet, pas d'estimations rédigées manuellement.

In [1]:
from __future__ import annotations

import ast
import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent

OUT = ROOT / 'outputs' / 'memoire_stats' / 'chapter3'
FIG = ROOT / 'rapport' / 'figures' / 'generated' / 'chapter3'
OUT.mkdir(parents=True, exist_ok=True)
FIG.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    'figure.figsize': (9, 5),
    'axes.grid': True,
    'grid.alpha': 0.25,
    'axes.titlesize': 12,
    'axes.labelsize': 10,
    'font.size': 10,
})

def load_env(path: Path) -> dict[str, str]:
    env = {}
    if not path.exists():
        return env
    for raw in path.read_text(encoding='utf-8').splitlines():
        line = raw.strip()
        if not line or line.startswith('#') or '=' not in line:
            continue
        key, value = line.split('=', 1)
        env[key.strip()] = value.strip().strip('"').strip("'")
    return env

ENV = {**load_env(ROOT / '.env'), **os.environ}

def save_table(df: pd.DataFrame, name: str) -> Path:
    path = OUT / f'{name}.csv'
    df.to_csv(path, index=False, encoding='utf-8-sig')
    return path

def save_fig(fig, name: str) -> tuple[Path, Path]:
    out_path = OUT / f'{name}.png'
    fig_path = FIG / f'{name}.png'
    fig.tight_layout()
    fig.savefig(out_path, dpi=180, bbox_inches='tight')
    fig.savefig(fig_path, dpi=180, bbox_inches='tight')
    plt.close(fig)
    return out_path, fig_path

print(f'Racine projet: {ROOT}')
print(f'Sorties tableaux: {OUT}')
print(f'Sorties figures LaTeX: {FIG}')

Racine projet: D:\DATA SCIENCES\SYSTEME-DE-RECOMMANDATION-HYBRIDE-
Sorties tableaux: D:\DATA SCIENCES\SYSTEME-DE-RECOMMANDATION-HYBRIDE-\outputs\memoire_stats\chapter3
Sorties figures LaTeX: D:\DATA SCIENCES\SYSTEME-DE-RECOMMANDATION-HYBRIDE-\rapport\figures\generated\chapter3


## 1. Chargement des données normalisées

In [2]:
DATA = ROOT / 'data' / 'processed'

paths = {
    'offres': DATA / 'offres_normalized.parquet',
    'candidats': DATA / 'candidats_normalized.parquet',
    'mepc_base': DATA / 'mepc_groupes_base.parquet',
    'ncf_detailles': DATA / 'ncf_dom_detailles.parquet',
    'mapping_isco_mepc_esco': DATA / 'mapping_isco_mepc_esco.parquet',
}

frames = {}
for name, path in paths.items():
    if path.exists():
        frames[name] = pd.read_parquet(path)
        print(f'{name}: {frames[name].shape[0]:,} lignes, {frames[name].shape[1]} colonnes')
    else:
        frames[name] = pd.DataFrame()
        print(f'{name}: fichier absent -> {path}')

offres = frames['offres']
candidats = frames['candidats']

offres: 7,861 lignes, 30 colonnes


candidats: 1,105 lignes, 20 colonnes
mepc_base: 209 lignes, 7 colonnes
ncf_detailles: 201 lignes, 6 colonnes
mapping_isco_mepc_esco: 19,987 lignes, 9 colonnes


## 2. Statistiques descriptives des données opérationnelles

In [3]:
def non_empty_rate(df: pd.DataFrame, col: str) -> float | None:
    if col not in df.columns or df.empty:
        return None
    s = df[col]
    ok = s.notna() & (s.astype(str).str.strip() != '')
    return round(float(ok.mean()), 4)

overview_rows = [
    {'bloc': 'Offres normalisées', 'unite': 'lignes', 'nombre': len(offres)},
    {'bloc': 'Candidats normalisés', 'unite': 'lignes', 'nombre': len(candidats)},
    {'bloc': 'Groupes de base MEPC', 'unite': 'lignes', 'nombre': len(frames['mepc_base'])},
    {'bloc': 'Domaines détaillés NCF', 'unite': 'lignes', 'nombre': len(frames['ncf_detailles'])},
    {'bloc': 'Mapping ISCO-MEPC-ESCO', 'unite': 'lignes', 'nombre': len(frames['mapping_isco_mepc_esco'])},
]
overview = pd.DataFrame(overview_rows)
save_table(overview, 'data_overview')
display(overview)

quality = pd.DataFrame([
    {'table': 'offres', 'champ': 'titre_poste', 'taux_non_vide': non_empty_rate(offres, 'titre_poste')},
    {'table': 'offres', 'champ': 'secteur_principal', 'taux_non_vide': non_empty_rate(offres, 'secteur_principal')},
    {'table': 'offres', 'champ': 'ville_principale', 'taux_non_vide': non_empty_rate(offres, 'ville_principale')},
    {'table': 'offres', 'champ': 'skills_list', 'taux_non_vide': non_empty_rate(offres, 'skills_list')},
    {'table': 'offres', 'champ': 'text_to_embed', 'taux_non_vide': non_empty_rate(offres, 'text_to_embed')},
    {'table': 'candidats', 'champ': 'candidat_id', 'taux_non_vide': non_empty_rate(candidats, 'candidat_id')},
    {'table': 'candidats', 'champ': 'ncf_niveau_final', 'taux_non_vide': non_empty_rate(candidats, 'ncf_niveau_final')},
    {'table': 'candidats', 'champ': 'metier_vise', 'taux_non_vide': non_empty_rate(candidats, 'metier_vise')},
    {'table': 'candidats', 'champ': 'text_to_embed', 'taux_non_vide': non_empty_rate(candidats, 'text_to_embed')},
]).dropna(subset=['taux_non_vide'])
save_table(quality, 'data_quality_non_empty_rates')
display(quality)

,bloc,unite,nombre
0,Offres normalisées,lignes,7861
1,Candidats normalisés,lignes,1105
2,Groupes de base MEPC,lignes,209
3,Domaines détaillés NCF,lignes,201
4,Mapping ISCO-MEPC-ESCO,lignes,19987


,table,champ,taux_non_vide
0,offres,titre_poste,1.0
1,offres,secteur_principal,1.0
2,offres,ville_principale,1.0
3,offres,skills_list,1.0
4,offres,text_to_embed,1.0
5,candidats,candidat_id,1.0
6,candidats,ncf_niveau_final,1.0
7,candidats,metier_vise,1.0
8,candidats,text_to_embed,1.0


## 3. Graphiques descriptifs pour le chapitre 3

In [4]:
generated_figures = []

def barh_counts(series: pd.Series, title: str, xlabel: str, name: str, top_n: int = 15):
    counts = series.dropna().astype(str).str.strip()
    counts = counts[counts != ''].value_counts().head(top_n).sort_values()
    df = counts.rename_axis('modalite').reset_index(name='nombre')
    save_table(df.sort_values('nombre', ascending=False), name)
    fig, ax = plt.subplots(figsize=(9, max(4.5, 0.32 * len(df))))
    ax.barh(df['modalite'], df['nombre'], color='#008997')
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel('')
    generated_figures.append(save_fig(fig, name))
    return df

if 'secteur_principal' in offres.columns:
    display(barh_counts(offres['secteur_principal'], 'Top secteurs des offres normalisées', 'Nombre d offres', 'offres_by_sector'))

if 'ville_principale' in offres.columns:
    display(barh_counts(offres['ville_principale'], 'Top localisations des offres normalisées', 'Nombre d offres', 'offres_by_city'))

ncf_col = next((c for c in ['ncf_niveau_final', 'ncf_code_niveau_etude', 'niveau_etude_raw'] if c in candidats.columns), None)
if ncf_col:
    display(barh_counts(candidats[ncf_col], f'Répartition des candidats par {ncf_col}', 'Nombre de candidats', 'candidats_by_ncf', top_n=12))

exp_col = next((c for c in ['experience_min_ans', 'experience_min', 'annees_experience_min'] if c in offres.columns), None)
if exp_col:
    exp = pd.to_numeric(offres[exp_col], errors='coerce').dropna()
    if len(exp) > 0:
        exp_summary = exp.describe(percentiles=[0.25, 0.5, 0.75, 0.9]).reset_index()
        exp_summary.columns = ['statistique', 'valeur']
        save_table(exp_summary, 'offres_experience_min_summary')
        fig, ax = plt.subplots(figsize=(8, 4.5))
        ax.hist(exp.clip(lower=0, upper=20), bins=20, color='#247e41', edgecolor='white')
        ax.set_title('Distribution de l expérience minimale demandée')
        ax.set_xlabel('Années d expérience minimale')
        ax.set_ylabel('Nombre d offres')
        generated_figures.append(save_fig(fig, 'offres_experience_min_hist'))
        display(exp_summary)

def parse_skills(value):
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return []
    if isinstance(value, (list, tuple, set)):
        return [str(x).strip() for x in value if str(x).strip()]
    text = str(value).strip()
    if not text:
        return []
    try:
        parsed = ast.literal_eval(text)
        if isinstance(parsed, (list, tuple, set)):
            return [str(x).strip() for x in parsed if str(x).strip()]
    except Exception:
        pass
    sep = ';' if ';' in text else ','
    return [x.strip() for x in text.split(sep) if x.strip()]

if 'skills_list' in offres.columns:
    skills = []
    for value in offres['skills_list']:
        skills.extend(parse_skills(value))
    if skills:
        skill_counts = pd.Series(skills).value_counts().head(20).sort_values()
        top_skills = skill_counts.rename_axis('competence').reset_index(name='nombre')
        save_table(top_skills.sort_values('nombre', ascending=False), 'top_skills_offres')
        fig, ax = plt.subplots(figsize=(9, 6))
        ax.barh(top_skills['competence'], top_skills['nombre'], color='#c42230')
        ax.set_title('Top 20 des compétences déclarées dans les offres')
        ax.set_xlabel('Fréquence')
        ax.set_ylabel('')
        generated_figures.append(save_fig(fig, 'top_skills_offres'))
        display(top_skills.sort_values('nombre', ascending=False).head(10))

print('Figures générées:')
for out_path, fig_path in generated_figures:
    print(f'- {fig_path.relative_to(ROOT)}')

,modalite,nombre
0,Gestion,145
1,Communication,204
2,Finance,209
3,Logistique,222
4,Banque,287
5,Autre,290
6,Marketing,296
7,Tic,306
8,Santé,307
9,Comptabilité,324


,modalite,nombre
0,Yaoundé - International,19
1,Bertoua,25
2,Cameroun,26
3,International,26
4,Douala - Yaoundé,28
5,Poli,30
6,Buéa,91
7,Kribi,99
8,Bamenda,100
9,Garoua,184


,modalite,nombre
0,9,1
1,3,55
2,8,60
3,1,71
4,6,76
5,4,193
6,7,290
7,5,359


,statistique,valeur
0,count,4721.0
1,mean,2.196992
2,std,2.450389
3,min,0.0
4,25%,1.0
5,50%,1.0
6,75%,3.0
7,90%,5.0
8,max,10.0


,competence,nombre
19,DistributionCommerce de grosCommerce de détail,524
18,INFORMATIQUE,316
17,TICTélécommunications,306
16,Autre,290
15,ONG,274
14,ADMINISTRATION,257
13,SANTÉ,200
12,BanqueServices financiersAssurance,194
11,FINANCE,163
10,COMMERCE,142


Figures générées:
- rapport\figures\generated\chapter3\offres_by_sector.png
- rapport\figures\generated\chapter3\offres_by_city.png
- rapport\figures\generated\chapter3\candidats_by_ncf.png
- rapport\figures\generated\chapter3\offres_experience_min_hist.png
- rapport\figures\generated\chapter3\top_skills_offres.png


## 4. Statistiques pgvector

In [5]:
pgvector_counts = pd.DataFrame()
pgvector_model_counts = pd.DataFrame()

try:
    import psycopg
    conn = psycopg.connect(
        host=ENV.get('PG_HOST', 'localhost'),
        port=int(ENV.get('PG_PORT', 5432)),
        dbname=ENV.get('PG_DB', ENV.get('POSTGRES_DB', 'test_kmer')),
        user=ENV.get('PG_USER', ENV.get('POSTGRES_USER', 'postgres')),
        password=ENV.get('PG_PASSWORD', ENV.get('POSTGRES_PASSWORD', '')),
    )
    with conn.cursor() as cur:
        cur.execute('''
            SELECT entity_kind::text AS entity_kind,
                   count(*)::int AS n,
                   count(neo4j_node_id)::int AS n_with_neo4j_id
            FROM embeddings
            GROUP BY 1
            ORDER BY n DESC
        ''')
        pgvector_counts = pd.DataFrame(cur.fetchall(), columns=['entity_kind', 'n', 'n_with_neo4j_id'])

        cur.execute('''
            SELECT model_id, count(*)::int AS n
            FROM embeddings
            GROUP BY model_id
            ORDER BY n DESC
        ''')
        pgvector_model_counts = pd.DataFrame(cur.fetchall(), columns=['model_id', 'n'])
    conn.close()

    save_table(pgvector_counts, 'pgvector_counts')
    save_table(pgvector_model_counts, 'pgvector_model_counts')
    display(pgvector_counts)
    display(pgvector_model_counts)

    if not pgvector_counts.empty:
        plot_df = pgvector_counts.sort_values('n')
        fig, ax = plt.subplots(figsize=(9, 5))
        ax.barh(plot_df['entity_kind'], plot_df['n'], color='#005aa0')
        ax.set_title('Volumes indexés dans pgvector par type d entité')
        ax.set_xlabel('Nombre d embeddings')
        ax.set_ylabel('')
        save_fig(fig, 'pgvector_counts_by_kind')
except Exception as exc:
    print(f'pgvector indisponible ou requête échouée: {exc}')

,entity_kind,n,n_with_neo4j_id
0,OFFRE_EMPLOI,15722,15722
1,COMPETENCE,13939,13939
2,METIER,3039,3039
3,CANDIDAT,1105,1105
4,GROUPE_BASE_MEPC,209,209
5,DOMAINE_DETAILLE_NCF,201,201


,model_id,n
0,all-MiniLM-L6-v2-ft-offres-cm,34215


## 5. Statistiques Neo4j

In [6]:
neo4j_node_counts = pd.DataFrame()
neo4j_rel_counts = pd.DataFrame()
total_nodes = None
total_rels = None

try:
    from neo4j import GraphDatabase

    driver = GraphDatabase.driver(
        ENV.get('NEO4J_URI', 'bolt://localhost:7687'),
        auth=(ENV.get('NEO4J_USER', 'neo4j'), ENV.get('NEO4J_PASSWORD', '')),
    )
    driver.verify_connectivity()
    database = ENV.get('NEO4J_DATABASE', 'neo4j')

    with driver.session(database=database) as session:
        try:
            node_rows = session.run('''
                MATCH (n)
                UNWIND labels(n) AS label
                RETURN label, count(*) AS n
                ORDER BY n DESC
            ''').data()
            total_nodes = session.run('MATCH (n) RETURN count(n) AS n').single()['n']
            neo4j_node_counts = pd.DataFrame(node_rows)
            save_table(neo4j_node_counts, 'neo4j_node_counts')
        except Exception as exc:
            print(f'Comptage des noeuds Neo4j ?chou?: {exc}')

        try:
            rel_rows = session.run('''
                MATCH ()-[r]->()
                RETURN type(r) AS relation, count(*) AS n
                ORDER BY n DESC
            ''').data()
            total_rels = session.run('MATCH ()-[r]->() RETURN count(r) AS n').single()['n']
            neo4j_rel_counts = pd.DataFrame(rel_rows)
            save_table(neo4j_rel_counts, 'neo4j_relation_counts')
        except Exception as exc:
            print(f'Comptage des relations Neo4j ?chou?: {exc}')

    driver.close()

    save_table(pd.DataFrame([
        {'metric': 'nodes', 'n': total_nodes},
        {'metric': 'relationships', 'n': total_rels},
    ]), 'neo4j_totals')

    print(f'Neo4j: {total_nodes if total_nodes is not None else "NA"} noeuds ; {total_rels if total_rels is not None else "NA"} relations')
    if not neo4j_node_counts.empty:
        display(neo4j_node_counts.head(20))
    if not neo4j_rel_counts.empty:
        display(neo4j_rel_counts.head(20))

    if not neo4j_node_counts.empty:
        plot_df = neo4j_node_counts.head(15).sort_values('n')
        fig, ax = plt.subplots(figsize=(9, 5.5))
        ax.barh(plot_df['label'], plot_df['n'], color='#f29700')
        ax.set_title('Noeuds Neo4j par label')
        ax.set_xlabel('Nombre de noeuds')
        ax.set_ylabel('')
        save_fig(fig, 'neo4j_nodes_by_label')

    if not neo4j_rel_counts.empty:
        plot_df = neo4j_rel_counts.head(15).sort_values('n')
        fig, ax = plt.subplots(figsize=(9, 5.5))
        ax.barh(plot_df['relation'], plot_df['n'], color='#414141')
        ax.set_title('Relations Neo4j par type')
        ax.set_xlabel('Nombre de relations')
        ax.set_ylabel('')
        save_fig(fig, 'neo4j_relationships_by_type')
except Exception as exc:
    print(f'Neo4j indisponible ou connexion ?chou?e: {exc}')

Comptage des relations Neo4j ?chou?: {neo4j_code: Neo.DatabaseError.Statement.ExecutionFailed} {message: org.neo4j.io.pagecache.CursorException: X1 Relationship(1067) is pointing to XD(76798) which has another owner 4735} {gql_status: 50N42} {gql_status_description: error: general processing exception - unexpected error. org.neo4j.io.pagecache.CursorException: X1 Relationship(1067) is pointing to XD(76798) which has another owner 4735}
Neo4j: 43008 noeuds ; NA relations


,label,n
0,GroupeCompétences,14579
1,Compétence,13939
2,OffreEmploi,7861
3,Métier,3039
4,Candidat,1105
5,Employeur,956
6,GroupeISCO,619
7,GroupeBaseMEPC,209
8,DomaineDétailléNCF,201
9,Secteur,190


## 6. Synthèse exportée pour le mémoire

In [7]:
summary = {
    'n_offres_normalisees': int(len(offres)),
    'n_candidats_normalises': int(len(candidats)),
    'n_mepc_groupes_base': int(len(frames['mepc_base'])),
    'n_ncf_domaines_detailles': int(len(frames['ncf_detailles'])),
    'n_pgvector_embeddings': int(pgvector_counts['n'].sum()) if not pgvector_counts.empty else None,
    'n_neo4j_nodes': int(neo4j_node_counts['n'].sum()) if not neo4j_node_counts.empty else None,
    'n_neo4j_relationships': int(neo4j_rel_counts['n'].sum()) if not neo4j_rel_counts.empty else None,
    'openrouter_model': ENV.get('OPENROUTER_MODEL', ''),
    'text2cypher_model': ENV.get('TEXT2CYPHER_MODEL', ''),
}

(OUT / 'implementation_summary.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
display(pd.DataFrame([summary]).T.rename(columns={0: 'valeur'}))

print('Sorties disponibles pour le chapitre 3:')
for path in sorted(OUT.glob('*')):
    print('-', path.relative_to(ROOT))
for path in sorted(FIG.glob('*.png')):
    print('-', path.relative_to(ROOT))

,valeur
n_offres_normalisees,7861
n_candidats_normalises,1105
n_mepc_groupes_base,209
n_ncf_domaines_detailles,201
n_pgvector_embeddings,34215
n_neo4j_nodes,43008
n_neo4j_relationships,None
openrouter_model,openai/gpt-oss-20b:free
text2cypher_model,neo4j/text2cypher-gemma-2-9b-it-finetuned-2024v1


Sorties disponibles pour le chapitre 3:
- outputs\memoire_stats\chapter3\candidats_by_ncf.csv
- outputs\memoire_stats\chapter3\candidats_by_ncf.png
- outputs\memoire_stats\chapter3\data_overview.csv
- outputs\memoire_stats\chapter3\data_quality_non_empty_rates.csv
- outputs\memoire_stats\chapter3\implementation_summary.json
- outputs\memoire_stats\chapter3\neo4j_node_counts.csv
- outputs\memoire_stats\chapter3\neo4j_nodes_by_label.png
- outputs\memoire_stats\chapter3\neo4j_totals.csv
- outputs\memoire_stats\chapter3\offres_by_city.csv
- outputs\memoire_stats\chapter3\offres_by_city.png
- outputs\memoire_stats\chapter3\offres_by_sector.csv
- outputs\memoire_stats\chapter3\offres_by_sector.png
- outputs\memoire_stats\chapter3\offres_experience_min_hist.png
- outputs\memoire_stats\chapter3\offres_experience_min_summary.csv
- outputs\memoire_stats\chapter3\pgvector_counts.csv
- outputs\memoire_stats\chapter3\pgvector_counts_by_kind.png
- outputs\memoire_stats\chapter3\pgvector_model_counts